In [124]:
 # # Import modules
import pandas as pd

In [125]:
# # set filter criteria
Filter_ttv_flag = 1
Filter_EarthRadius_lower = 3.0
Filter_EarthRadius_upper = 8.5
Filter_Vmag = 13.0

In [134]:
# ratio of radii . Earth to Jupiter. Carole Haswell Book. used to convert Jupiter Radii to Earth equivalents
radius_earth_m = 6.37e6
radius_jupiter_m = 7.15e7
ratio_earth_to_jupiter = radius_earth_m/radius_jupiter_m

In [126]:
# # Define planet classification ranges
df_NASA_ranges = pd.DataFrame({"lower":[0,0.5,1.0,1.75,3.5,6.0,14.3],\
                               "upper": [0.5,1.0,1.75,3.5,6.0,14.3,9999],\
                               "classification": ["1.Radius below lower bound","2.Rocky planet","3.Super Earth","4.Sub Neptune","5.Neptune","6.Jupiters","7.Radius above upper bound"] \
                              })
intervals = pd.IntervalIndex.from_arrays(df_NASA_ranges["lower"],df_NASA_ranges["upper"],closed="both")

In [127]:
# # Function to set planet classifications
def classify_exoplanet(radius):
    match = df_NASA_ranges.loc[intervals.contains(radius),"classification"]
    return match.iloc[0] if not match.empty else None

In [139]:
# Define function to clean exoplanet names
def clean_name(s):
    return (
        s.str.lower()
         .str.replace(" ", "", regex=False)
         .str.replace("-", "", regex=False)
         .str.replace("_", "", regex=False)
    )

In [132]:
# # Load raw data files
df_NASA_data = pd.read_csv("/Users/danielbhuglah/Downloads/PS_2026.01.05_06.10.12.CSV", skiprows=292)
df_EU_data = pd.read_csv("/Users/danielbhuglah/Downloads/exoplanet.eu_catalog_02-01-26_16_43_00.CSV")

/var/folders/z6/hb5zbv6x4y3216b82yynqb740000gn/T/ipykernel_19123/350035016.py:2: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_NASA_data = pd.read_csv("/Users/danielbhuglah/Downloads/PS_2026.01.05_06.10.12.CSV", skiprows=292)


In [129]:
# # NASA Processing
# NASA - only select exoplanet data where 
# i) default flag is 1 and ii) the planet has been confirmed and iii) TTV flag is 1
df_NASA_data_filtered = df_NASA_data[(df_NASA_data["default_flag"] == 1) &
                            (df_NASA_data["soltype"] == "Published Confirmed") &
                            (df_NASA_data["ttv_flag"] == Filter_ttv_flag) &
                            (df_NASA_data["pl_rade"] >= Filter_EarthRadius_lower) &
                            (df_NASA_data["pl_rade"] <= Filter_EarthRadius_upper) &
                            (df_NASA_data["sy_vmag"] <= Filter_Vmag)
                            ]


In [130]:
# # Set exoplanet classification. Copy the filtered DF to ensure no issues with updating. 
df_NASA_data_filtered_V2 = df_NASA_data_filtered.copy()
df_NASA_data_filtered_V2["Exoplanet_class"] = df_NASA_data_filtered_V2["pl_rade"].apply(classify_exoplanet)

In [131]:
df_NASA_data_filtered_V2.to_excel("/Users/danielbhuglah/Downloads/NASA_data_filtered_V2.xlsx",index=False)

In [135]:
# # EU Processing
# Raw data file loaded earlier in code
# Covert the eu planet radius (in Jupiter eqivalents) into earth radius equivalents. used to do classifications and normals with NASA archives
df_EU_data["radius_earth"]= df_EU_data["radius"]/ ratio_earth_to_jupiter

In [137]:
# use the earth radius equivalents to assign exoplanet classificaitons
df_EU_data["Exoplanet_class"] = df_EU_data["radius_earth"].apply(classify_exoplanet)

In [138]:
df_EU_data

,name,planet_status,mass,mass_error_min,mass_error_max,mass_sini,mass_sini_error_min,mass_sini_error_max,radius,radius_error_min,...,star_age_error_min,star_age_error_max,star_teff,star_teff_error_min,star_teff_error_max,star_detected_disc,star_magnetic_field,star_alternate_names,radius_earth,Exoplanet_class
0,109 Psc b,Confirmed,5.743,0.28900,1.01100,6.38300,0.07800,0.07800,1.152,NaN,...,0.60,0.60,5600.0,80.000,80.000,NaN,NaN,HD 10697,12.930612,6.Jupiters
1,112 Psc b,Confirmed,NaN,0.00500,0.00400,0.03300,0.00500,0.00400,NaN,NaN,...,NaN,NaN,5986.0,105.437,105.437,NaN,NaN,HD 12235,NaN,None
2,112 Psc c,Confirmed,9.866,1.78100,3.19000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,5986.0,105.437,105.437,NaN,NaN,HD 12235,NaN,None
3,11 UMi b,Confirmed,NaN,1.10000,1.10000,11.08730,1.10000,1.10000,NaN,NaN,...,0.54,0.54,4340.0,70.000,70.000,NaN,NaN,NaN,NaN,None
4,14 And Ab,Confirmed,NaN,0.23000,0.23000,4.68400,0.23000,0.23000,NaN,NaN,...,NaN,NaN,4813.0,20.000,20.000,NaN,NaN,NaN,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6194,YBP 1514 b,Confirmed,NaN,0.11000,0.11000,0.40000,0.11000,0.11000,NaN,NaN,...,NaN,NaN,5725.0,45.000,45.000,NaN,NaN,"Cl* NGC 2682 YBP 1514, EPIC 211416296",NaN,None
6195,YBP 401 b,Confirmed,NaN,0.05000,0.05000,0.46000,0.05000,0.05000,NaN,NaN,...,NaN,NaN,6165.0,64.000,64.000,NaN,NaN,"Cl* NGC 2682 YBP 401, NGC 2682 122",NaN,None
6196,YZ Cet b,Confirmed,NaN,0.00028,0.00028,0.00220,0.00028,0.00028,NaN,NaN,...,0.60,0.60,3056.0,60.000,60.000,NaN,NaN,NaN,NaN,None
6197,YZ Cet c,Confirmed,NaN,0.00035,0.00035,0.00359,0.00035,0.00035,NaN,NaN,...,0.60,0.60,3056.0,60.000,60.000,NaN,NaN,NaN,NaN,None


In [140]:
# clean names by removing blanks, dashes and underscores
df_EU_data["clean_name"] = clean_name(df_EU_data["name"])
df_NASA_data["clean_name"] = clean_name(df_NASA_data["pl_name"])

In [142]:
# explode the alternate name list from eu data and create one row per exploded name including the orginal data row
df_EU_data["alt_list"]= df_EU_data["alternate_names"].str.split(",")
df_EU_data_exploded = df_EU_data.explode("alt_list")
df_EU_data_exploded["clean_alt"] = clean_name(df_EU_data_exploded["alt_list"])

In [143]:
df_EU_data_exploded

,name,planet_status,mass,mass_error_min,mass_error_max,mass_sini,mass_sini_error_min,mass_sini_error_max,radius,radius_error_min,...,star_teff_error_min,star_teff_error_max,star_detected_disc,star_magnetic_field,star_alternate_names,radius_earth,Exoplanet_class,clean_name,alt_list,clean_alt
0,109 Psc b,Confirmed,5.743,0.28900,1.01100,6.38300,0.07800,0.07800,1.152,NaN,...,80.000,80.000,NaN,NaN,HD 10697,12.930612,6.Jupiters,109pscb,HD 10697 b,hd10697b
1,112 Psc b,Confirmed,NaN,0.00500,0.00400,0.03300,0.00500,0.00400,NaN,NaN,...,105.437,105.437,NaN,NaN,HD 12235,NaN,None,112pscb,HD 12235 b,hd12235b
2,112 Psc c,Confirmed,9.866,1.78100,3.19000,NaN,NaN,NaN,NaN,NaN,...,105.437,105.437,NaN,NaN,HD 12235,NaN,None,112pscc,HD 12235 c,hd12235c
3,11 UMi b,Confirmed,NaN,1.10000,1.10000,11.08730,1.10000,1.10000,NaN,NaN,...,70.000,70.000,NaN,NaN,NaN,NaN,None,11umib,NaN,NaN
4,14 And Ab,Confirmed,NaN,0.23000,0.23000,4.68400,0.23000,0.23000,NaN,NaN,...,20.000,20.000,NaN,NaN,NaN,NaN,None,14andab,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6195,YBP 401 b,Confirmed,NaN,0.05000,0.05000,0.46000,0.05000,0.05000,NaN,NaN,...,64.000,64.000,NaN,NaN,"Cl* NGC 2682 YBP 401, NGC 2682 122",NaN,None,ybp401b,NaN,NaN
6196,YZ Cet b,Confirmed,NaN,0.00028,0.00028,0.00220,0.00028,0.00028,NaN,NaN,...,60.000,60.000,NaN,NaN,NaN,NaN,None,yzcetb,NaN,NaN
6197,YZ Cet c,Confirmed,NaN,0.00035,0.00035,0.00359,0.00035,0.00035,NaN,NaN,...,60.000,60.000,NaN,NaN,NaN,NaN,None,yzcetc,NaN,NaN
6198,YZ Cet d,Confirmed,NaN,0.00038,0.00038,0.00343,0.00038,0.00038,NaN,NaN,...,60.000,60.000,NaN,NaN,NaN,NaN,None,yzcetd,GJ 54.1,gj54.1


In [145]:
# setup columns to be pulled in from NASA data
NASA_cols = ["clean_name","ttv_flag"]
# merge the EU data with NASA equivalents - left merge
df_merged_main = df_EU_data.merge(df_NASA_data,on="clean_name",how="left",suffixes=("","_nasa"))

In [146]:
df_merged_main

,name,planet_status,mass,mass_error_min,mass_error_max,mass_sini,mass_sini_error_min,mass_sini_error_max,radius,radius_error_min,...,rowupdate,pl_pubdate,releasedate,pl_nnotes,st_nphot,st_nrvc,st_nspec,pl_nespec,pl_ntranspec,pl_ndispec
0,109 Psc b,Confirmed,5.743,0.28900,1.01100,6.38300,0.07800,0.07800,1.152,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,112 Psc b,Confirmed,NaN,0.00500,0.00400,0.03300,0.00500,0.00400,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,112 Psc c,Confirmed,9.866,1.78100,3.19000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,11 UMi b,Confirmed,NaN,1.10000,1.10000,11.08730,1.10000,1.10000,NaN,NaN,...,2018-04-25,2009-10,2014-05-14,0.0,1.0,1.0,0.0,0.0,0.0,0.0
4,11 UMi b,Confirmed,NaN,1.10000,1.10000,11.08730,1.10000,1.10000,NaN,NaN,...,2018-09-04,2017-03,2018-09-06,0.0,1.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32802,YZ Cet c,Confirmed,NaN,0.00035,0.00035,0.00359,0.00035,0.00035,NaN,NaN,...,2017-08-22,2017-09,2017-08-24,2.0,1.0,0.0,0.0,0.0,0.0,0.0
32803,YZ Cet c,Confirmed,NaN,0.00035,0.00035,0.00359,0.00035,0.00035,NaN,NaN,...,2019-03-25,2018-09,2019-03-28,2.0,1.0,0.0,0.0,0.0,0.0,0.0
32804,YZ Cet d,Confirmed,NaN,0.00038,0.00038,0.00343,0.00038,0.00038,NaN,NaN,...,2019-03-25,2018-09,2019-03-28,0.0,1.0,0.0,0.0,0.0,0.0,0.0
32805,YZ Cet d,Confirmed,NaN,0.00038,0.00038,0.00343,0.00038,0.00038,NaN,NaN,...,2017-08-22,2017-09,2017-08-24,0.0,1.0,0.0,0.0,0.0,0.0,0.0
